Import the libraries math (for square root and absolute value) and pandas and numpy for reading csv files.

In [1]:
import math
import pandas as pd
import numpy as np

Make an untagged vector, point, and two tagged vectors, data1 and data2:

In [2]:
point = [1, 0, 0, '?'] #(an unknown tag)
data1 = [1, 1, 1, 'M']
data2 = [1, 3, 0, 'F']


Write code to separate the data (X) from the tag (Y).  Your output should be:

The vector [1, 1, 1] has tag M

The vector [1, 3, 0] has tag F


In [4]:
print("The vector" , data1[0:-1], "has tag", data1[-1])
print("The vector" , data2[0:-1], "has tag", data2[-1])

The vector [1, 1, 1] has tag M
The vector [1, 3, 0] has tag F


Now let's classify the point as either M or F.  We'll do this by setting k = 1 and using the Euclidean distance.  We'll define that as:

In [5]:
def euclideanDistance(instance1, instance2, length):
   distance = 0
   for x in range(length):
         #print ('x is ' , x)
         num1=float(instance1[x])
         num2=float(instance2[x])
         distance = distance + pow(num1-num2, 2)
   return math.sqrt(distance)


Now, find out the distance between data1 and point and data2 and point. Output what the tag should be using euclideanDistance.

In [6]:
dist1 = euclideanDistance(data1, point, 3)
dist2 = euclideanDistance(data2, point, 3)
print("Distance between", data1[0:-1], "and", point[0:-1], "is", dist1)
print("Distance between", data2[0:-1], "and", point[0:-1], "is", dist2)

# with k = 1 the point gets the tag of its single nearest neighbor
if dist1 < dist2:
    print("The point", point[0:-1], "is classified as", data1[-1])
else:
    print("The point", point[0:-1], "is classified as", data2[-1])

Distance between [1, 1, 1] and [1, 0, 0] is 1.4142135623730951
Distance between [1, 3, 0] and [1, 0, 0] is 3.0
The point [1, 0, 0] is classified as M


Now, let's get more data from a file, myFile.csv. This code will read it for us:

In [7]:
url = 'https://github.com/rosenfa/ai/blob/master/myFile.csv?raw=true'
df=pd.read_csv(url,  header=0)
#put data in dataset without header line
dataset = np.array(df)

In [8]:
dataset

array([[0, 1, 2, 'F'],
       [1, 5, 6, 'F'],
       [1, 40, 1, 'M'],
       [2, 2, 2, 'F'],
       [0, 1, 20, 'M'],
       [1, 5, 6, 'M'],
       [1, 4, 1, 'M'],
       [2, 22, 2, 'F'],
       [2, 23, 2, 'M'],
       [2, 24, 2, 'M']], dtype=object)

Show that you understand what you read by:

1. Printing the first two vectors in the file
2. Printing the Euclidean distance between those two vectors

Here is some code which I think might help get you on your way!

In [9]:
print(dataset[0])
print(dataset[1])
print(euclideanDistance(dataset[0], dataset[1], 3))

[0 1 2 'F']
[1 5 6 'F']
5.744562646538029


Now assume you have a new value for point:

[0,0,100]

How would you classify this vector using the Euclidean distance function given all of the vectors in the file?

In order to help here is a hint:  

We suggest defining some type of data structure to store different vectors' distances and their tags like this:

In [10]:
point=[0,0,100]
class distClass:
    dist = -1 #distance of current point from test point
    tag = '-' #tag of current point


You can then add vector distances like this:

In [11]:
eucDistances = [] # list of distances, will hold objects of type distClass

temp=dataset[1]
label=temp[-1]
d=euclideanDistance(point,temp,3)
print("The distances between " , point , " and " , temp,  " is " , str(d))
print(" and the label is " + label)
obj = distClass() #one record's distance and tag
obj.dist=d
obj.tag=label
eucDistances.append(obj)


The distances between  [0, 0, 100]  and  [1 5 6 'F']  is  94.13819628609845
 and the label is F


In [12]:
eucDistances[0].dist

94.13819628609845

and sort them like this:

In [13]:
eucDistances.sort(key=lambda x: x.dist)

Questions:

1. What is the label for point if k=1?
2. What is the label for point if k=3?
3. Would the result be different if we used a different distance function like Hamming or Manhattan?

In [14]:
def hammingDistance(instance1, instance2, length):
   """Number of features (positions) in which the two vectors differ."""
   distance = 0
   for x in range(length):
         if float(instance1[x]) != float(instance2[x]):
               distance = distance + 1
   return distance

def manhattanDistance(instance1, instance2, length):
   """Sum of the absolute differences of the features."""
   distance = 0
   for x in range(length):
         distance = distance + abs(float(instance1[x]) - float(instance2[x]))
   return distance

def knnClassify(trainSet, testVector, k, distanceFunction):
   """Return the majority tag among the k vectors in trainSet closest to testVector.

   The last column of every training vector is its tag, so only the first
   len(trainVector) - 1 columns are used when measuring distance.
   """
   numFeatures = len(trainSet[0]) - 1
   distances = [] # list of distances, will hold objects of type distClass
   for trainVector in trainSet:
         obj = distClass()
         obj.dist = distanceFunction(testVector, trainVector, numFeatures)
         obj.tag = trainVector[-1]
         distances.append(obj)
   # list.sort() is stable, so equal distances keep their order in the training file
   distances.sort(key=lambda x: x.dist)

   # majority vote among the k nearest neighbors
   votes = {}
   for obj in distances[:k]:
         votes[obj.tag] = votes.get(obj.tag, 0) + 1
   return max(votes, key=votes.get)

distanceFunctions = {
   "Euclidean": euclideanDistance,
   "Hamming": hammingDistance,
   "Manhattan": manhattanDistance,
}

point = [0, 0, 100]
for name, distanceFunction in distanceFunctions.items():
   for k in [1, 3]:
         print(f"{name:<9} k={k}: {point} is classified as {knnClassify(dataset, point, k, distanceFunction)}")

Euclidean k=1: [0, 0, 100] is classified as M
Euclidean k=3: [0, 0, 100] is classified as M
Hamming   k=1: [0, 0, 100] is classified as F
Hamming   k=3: [0, 0, 100] is classified as F
Manhattan k=1: [0, 0, 100] is classified as M
Manhattan k=3: [0, 0, 100] is classified as F


Answers:

1. M
2. M
3. Yes:
   - Hamming:
      - 1: F
      - 3: F
   - Manhattan:
      - 1: M
      - 3: F

Up until here is a simplified version of the homework.
Below here is the work for the part we will be checking as the basis of your grade:

Now let's look at some bigger files:

1. mytrain.csv (for training the model)
2. mytest.csv (for testing)


In [15]:
url = 'https://github.com/rosenfa/ai/blob/master/mytrain.csv?raw=true'
train_data = np.array(pd.read_csv(url,  header=0))
url = 'https://github.com/rosenfa/ai/blob/master/mytest.csv?raw=true'
test_data = np.array(pd.read_csv(url,  header=0))

In [16]:
print(train_data.shape)# number of records and features
print(train_data)

(100, 30)
[[0.317251 0.540016 0.150514 ... 0.0 0.212007 'F']
 [3.882181 0.0 0.0 ... 0.0 0.0 'F']
 [1.790708 0.0 0.0 ... 0.0 0.0 'F']
 ...
 [0.0 0.0 0.0 ... 0.0 0.0 'M']
 [0.0 0.0 0.0 ... 0.0 0.0 'M']
 [0.0 0.0 0.0 ... 0.0 0.0 'F']]


In [17]:
print(test_data.shape)# number of records and features
print(test_data)

(100, 30)
[[2.85985 0.66275 0.0 ... 0.995289 0.0 'F']
 [0.0 12.998726 0.0 ... 0.0 0.0 'F']
 [13.231318 0.0 0.0 ... 0.0 0.0 'F']
 ...
 [0.0 0.0 0.0 ... 0.0 0.0 'M']
 [1.688611 0.0 3.204529 ... 0.0 0.0 'M']
 [0.0 0.0 0.0 ... 0.0 0.0 'M']]


I hope by now you understand where we are going with this :)

Now implement the knn code with 3 different values for k:
1. k = 1
2. k = 7
3. k = 15

and at first use the Euclidean distance.

Tie-breaking: If several training vectors are at the same distance from a test vector, keep them in the order they appear in the training file. Python's `list.sort()` does this automatically. If you use `np.argsort`, pass `kind='stable'`. With two tags and odd k, there are no ties in the majority vote.

Classify each of the vectors in the test_data dataset using the training data from train_data.  Which value for k did the best?  What *accuracy* did it give you?

Now see if using Hamming or Manhattan distance give any better results for the same values of k.  

Once you are done, you should have a total of 9 different results:

1. Three results for the different value of k using the Euclidean Distance
2. Three results for the different value of k using the Hamming Distance
3. Three results for the different value of k using the Manhattan Distance

Hint: I strongly suggest you base yourself on the code you've seen until this point so you don't have to reinvent the wheel!

In [18]:
kValues = [1, 7, 15]

# predictions[(distance name, k)] = list of predicted tags, one per test vector
predictions = {}
for name, distanceFunction in distanceFunctions.items():
   for k in kValues:
         predictions[(name, k)] = [knnClassify(train_data, testVector, k, distanceFunction)
                                   for testVector in test_data]

trueTags = list(test_data[:, -1])

def accuracy(trueTags, predictedTags):
   """Fraction of test vectors whose predicted tag equals the true tag."""
   correct = sum(1 for t, p in zip(trueTags, predictedTags) if t == p)
   return correct / len(trueTags)

for (name, k), predictedTags in predictions.items():
   print(f"{name:<9} k={k:<2}: accuracy = {accuracy(trueTags, predictedTags):.2f}")

Euclidean k=1 : accuracy = 0.50
Euclidean k=7 : accuracy = 0.74
Euclidean k=15: accuracy = 0.70
Hamming   k=1 : accuracy = 0.61
Hamming   k=7 : accuracy = 0.55
Hamming   k=15: accuracy = 0.57
Manhattan k=1 : accuracy = 0.61
Manhattan k=7 : accuracy = 0.63
Manhattan k=15: accuracy = 0.69


Answers: With Euclidian distance, K=7 give the best accuracy with 0.74. neither Hamming or Manhattan can beat that. under any k.

Precision, Recall and F-measure:

Accuracy alone doesn't tell the whole story, so for each of the 9 results above you must also report the precision, recall, and f-measure.

Compute them separately for each tag in the dataset. For a given tag T:

Precision(T) = (number of test vectors correctly classified as T) / (number of test vectors your model classified as T)

Recall(T) = (number of test vectors correctly classified as T) / (number of test vectors whose true tag is T)

F-measure(T) = 2 * Precision(T) * Recall(T) / (Precision(T) + Recall(T))

Once you are done, each of your 9 results should include accuracy, precision, recall, and f-measure.

Then answer: Is the combination of k and distance function that gave the best accuracy also the best for precision, recall, and f-measure? If not, explain the difference.

In [19]:
def tagMetrics(trueTags, predictedTags, tag):
   """Precision, recall and F-measure for a single tag."""
   correctAsTag = sum(1 for t, p in zip(trueTags, predictedTags) if t == tag and p == tag)
   predictedAsTag = sum(1 for p in predictedTags if p == tag)
   actuallyTag = sum(1 for t in trueTags if t == tag)

   precision = correctAsTag / predictedAsTag if predictedAsTag > 0 else 0
   recall = correctAsTag / actuallyTag if actuallyTag > 0 else 0
   fMeasure = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0
   return precision, recall, fMeasure

tags = sorted(set(trueTags))
rows = []
for (name, k), predictedTags in predictions.items():
   row = {"distance": name, "k": k, "accuracy": accuracy(trueTags, predictedTags)}
   for tag in tags:
         precision, recall, fMeasure = tagMetrics(trueTags, predictedTags, tag)
         row[f"precision({tag})"] = precision
         row[f"recall({tag})"] = recall
         row[f"f-measure({tag})"] = fMeasure
   rows.append(row)

results = pd.DataFrame(rows)
results.round(3)

,distance,k,accuracy,precision(F),recall(F),f-measure(F),precision(M),recall(M),f-measure(M)
0,Euclidean,1,0.50,0.528,0.704,0.603,0.429,0.261,0.324
1,Euclidean,7,0.74,0.719,0.852,0.780,0.778,0.609,0.683
2,Euclidean,15,0.70,0.643,1.000,0.783,1.000,0.348,0.516
3,Hamming,1,0.61,0.582,0.981,0.731,0.889,0.174,0.291
4,Hamming,7,0.55,0.548,0.944,0.694,0.571,0.087,0.151
5,Hamming,15,0.57,0.557,1.000,0.715,1.000,0.065,0.122
6,Manhattan,1,0.61,0.612,0.759,0.678,0.606,0.435,0.506
7,Manhattan,7,0.63,0.613,0.852,0.713,0.680,0.370,0.479
8,Manhattan,15,0.69,0.635,1.000,0.777,1.000,0.326,0.492


Answer: No. the test set has more F than M, so for large K (like k = 15) the model is biased towards predicting F, which will increase recall(F) and precision(M). However, this is not a real improvement, just a bias towards selecting F. (Also f-measure(F) is slightly higher with euclidean distance and k=15, but only slightly, and f-measure(M) decreases total f-measure (F+M) is still higher with k=7)

Grade Basis:

80% for correct answers, including accuracy, precision, recall, and f-measure for all 9 results (and yes, there are possibilities that multiple answers are correct-- especially for cases of ties).

20% : Documentation and easily readable code

Please publish your final Notebook in your Github directory.

The homework assignment is due by November 2nd at 11:59 PM.
